In [1]:
import pandas as pd

In [20]:
def extrair_pipes(arquivo_inp):
    with open(arquivo_inp, 'r', encoding='utf-8') as f:
        linhas = f.readlines()
    
    pipes_data = []
    coletando = False  # Flag para indicar quando estamos na seção [PIPES]

    for linha in linhas:
        linha = linha.strip()
        if linha.startswith("[PIPES]"):
            coletando = True
            continue
        if coletando:
            if linha == "" or linha.startswith("["):  # Sai da seção se encontrar outra seção
                break
            if not linha.startswith(";"):  # Ignora comentários
                pipes_data.append(linha.split())

    # Criando DataFrame
    colunas = ["ID", "Node1", "Node2", "Length", "Diameter", "Roughness", "MinorLoss", "Status", "empty"]
    df = pd.DataFrame(pipes_data, columns=colunas)

    return df

# def extrair_pipes(arquivo_inp):
#     with open(arquivo_inp, 'r', encoding='utf-8') as f:
#         linhas = f.readlines()
    
#     pipes_data = []
#     coletando = False

#     for linha in linhas:
#         linha = linha.strip()

#         if linha.startswith("[PIPES]"):
#             coletando = True
#             continue

#         if coletando:
#             if linha == "" or linha.startswith("["):
#                 break

#             if not linha.startswith(";"):
#                 valores = linha.split()

#                 # Garante 8 colunas (Status opcional)
#                 if len(valores) == 7:
#                     valores.append("")  # Status vazio

#                 pipes_data.append(valores)

#     colunas = [
#         "ID", "Node1", "Node2", "Length",
#         "Diameter", "Roughness", "MinorLoss", "Status"
#     ]

#     df = pd.DataFrame(pipes_data, columns=colunas)

#     # Converte numéricos
#     for col in ["Length", "Diameter", "Roughness", "MinorLoss"]:
#         df[col] = pd.to_numeric(df[col], errors="coerce")

#     return df


def extrair_cordenadas(arquivo_inp):
    with open(arquivo_inp, 'r', encoding='utf-8') as f:
        linhas = f.readlines()
    
    pipes_data = []
    coletando = False  # Flag para indicar quando estamos na seção [PIPES]

    for linha in linhas:
        linha = linha.strip()
        if linha.startswith("[COORDINATES]"):
            coletando = True
            continue
        if coletando:
            if linha == "" or linha.startswith("["):  # Sai da seção se encontrar outra seção
                break
            if not linha.startswith(";"):  # Ignora comentários
                pipes_data.append(linha.split())

    # Criando DataFrame
    colunas = ["Node", "X-Coord", "Y-Coord"]
    df = pd.DataFrame(pipes_data, columns=colunas)

    return df

def getNodeCoordinates(df_pipes, df_coordenadas):
    # Converte colunas numéricas para float (evita erro na hora de juntar os DataFrames)
    df_coordenadas[["X-Coord", "Y-Coord"]] = df_coordenadas[["X-Coord", "Y-Coord"]].astype(float)

    # Junta as coordenadas dos nós iniciais (Node1) e finais (Node2)
    df_pipes = df_pipes.merge(df_coordenadas, left_on="Node1", right_on="Node", how="left").rename(
        columns={"X-Coord": "x_start", "Y-Coord": "y_start"}
    ).drop(columns=["Node"])

    df_pipes = df_pipes.merge(df_coordenadas, left_on="Node2", right_on="Node", how="left").rename(
        columns={"X-Coord": "x_end", "Y-Coord": "y_end"}
    ).drop(columns=["Node"])

    return df_pipes[["ID", "Node1", "Node2", "x_start", "y_start", "x_end", "y_end"]]

def setLinkRoughnessCoeff(df_pipes, list_pipe_id, new_roughness):
    for id in list_pipe_id:
        df_pipes.loc[df_pipes["ID"] == id, "Roughness"] = new_roughness

    return df_pipes

def update_pipes_inp(arquivo_inp, df_pipes):
    with open(arquivo_inp, 'r', encoding='utf-8') as f:
        linhas = f.readlines()

    nova_secao_pipes = []
    coletando = False

    for linha in linhas:
        linha_strip = linha.strip()

        if linha_strip.startswith("[PIPES]"):
            coletando = True
            nova_secao_pipes.append("[PIPES]\n")
            nova_secao_pipes.append(";ID              	Node1           	Node2           	Length      	Diameter    	Roughness   	MinorLoss   	Status\n")
            continue
        elif coletando and (linha_strip == "" or linha_strip.startswith("[")):  
            coletando = False

            # Adiciona os pipes atualizados com formatação correta
            for _, row in df_pipes.iterrows():
                nova_linha = f"{row['ID']: <18}\t{row['Node1']: <16}\t{row['Node2']: <16}\t{row['Length']: <14}\t{row['Diameter']: <12}\t{row['Roughness']: <12}\t{row['MinorLoss']: <12}\t{row['Status']: <10};\n"
                nova_secao_pipes.append(nova_linha)
        
        if not coletando:
            nova_secao_pipes.append(linha)

    # Escrevendo de volta no arquivo .inp
    with open(arquivo_inp, 'w', encoding='utf-8') as f:
        f.writelines(nova_secao_pipes)

def adjust_roughness_with_limit(df_pipes, list_pipe_id, new_adjust_roughness, limit=40):
    """
    Ajusta a rugosidade dos pipes, removendo um valor específico,
    mas garantindo que ela nunca fique abaixo do limite definido.

    Parâmetros:
        df_pipes (DataFrame): DataFrame contendo os dados dos pipes.
        list_pipe_id (list): Lista de IDs dos pipes que terão a rugosidade ajustada.
        new_adjust_roughness (float): Valor a ser subtraído da rugosidade.
        limit (float, opcional): Valor mínimo permitido para a rugosidade. Padrão é 40.

    Retorna:
        DataFrame atualizado com os novos valores de rugosidade.
    """

    index = df_pipes[df_pipes["ID"].isin(list_pipe_id)].index
    
    for i in index:
        roughness = df_pipes.loc[i, "Roughness"]
        new_roughness = max(roughness - new_adjust_roughness, limit)  # Garante que não passe do limite
        df_pipes.loc[i, "Roughness"] = new_roughness

    return df_pipes



In [21]:

arquivo = r"Epanet\Sao Sebastiao\Novo plano\SSB - 32 PARA 63.inp"

df_pipes = extrair_pipes(arquivo)
df_coordenadas = extrair_cordenadas(arquivo)
df_pipes_com_coords = getNodeCoordinates(df_pipes, df_coordenadas)
df_pipes_com_coords

,ID,Node1,Node2,x_start,y_start,x_end,y_end
0,0,7867,7504,204358.477,8239447.250,204359.759,8239444.241
1,1,7664,7665,204549.212,8238691.322,204553.793,8238691.466
2,2,7663,7664,204537.313,8238691.196,204549.212,8238691.322
3,3,7662,7663,204530.853,8238691.168,204537.313,8238691.196
4,4,7661,7662,204524.000,8238690.675,204530.853,8238691.168
...,...,...,...,...,...,...,...
7661,361,169,2499,204380.485,8238760.000,204382.034,8238760.655
7662,634,2348,3736,204930.000,8238872.286,204936.909,8238877.955
7663,672,5620,5619,205438.003,8239071.652,205442.504,8239047.261
7664,10,6938,10000,202524.226,8240033.576,202518.905,8240025.391


In [22]:
df_pipes

,ID,Node1,Node2,Length,Diameter,Roughness,MinorLoss,Status,empty
0,0,7867,7504,3.2711,110.0000,140.0000,0.0000,Open,;
1,1,7664,7665,4.5834,250.0000,130,0.0000,Open,;
2,2,7663,7664,11.8994,250.0000,130,0.0000,Open,;
3,3,7662,7663,6.4606,250.0000,130,0.0000,Open,;
4,4,7661,7662,6.8711,250.0000,130,0.0000,Open,;
...,...,...,...,...,...,...,...,...,...
7661,361,169,2499,1.0000,32.0000,110,0.0000,Open,;
7662,634,2348,3736,1,200,137,0,Open,;
7663,672,5620,5619,1,90,135,0,Open,;
7664,10,6938,10000,1,300,135,0,Open,;


In [26]:
analise = df_pipes[df_pipes['Diameter']==32]
analise

,ID,Node1,Node2,Length,Diameter,Roughness,MinorLoss,Status,empty
13,16,7354,7396,21.1210,32.0,120.0000,0.0000,Open,;
14,17,1148,7652,1.8976,32.0,120.0000,0.0000,Open,;
43,62,8085,8086,14.8812,32.0,140.0000,0.0000,Open,;
44,63,7975,8085,20.4071,32.0,140.0000,0.0000,Open,;
136,161,6371,4824,5.5720,32.0,140.0000,0.0000,Open,;
...,...,...,...,...,...,...,...,...,...
7620,261,6369,2269,1.0000,32.0,140.0000,0.0000,Open,;
7634,34,3717,565,10.0000,32.0,130,0.0000,Open,;
7649,626,164,3934,1.0000,32.0,130,0.0000,Open,;
7656,202,6884,2359,0.1000,32.0,130.0000,0.0000,Open,;


In [27]:
analise['Length'].sum()

62831.2614

In [5]:
df_pipes

,ID,Node1,Node2,Length,Diameter,Roughness,MinorLoss,Status,empty
0,0,7867,7504,3.2711,110.0000,140.0000,0.0000,Open,;
1,1,7664,7665,4.5834,250.0000,130,0.0000,Open,;
2,2,7663,7664,11.8994,250.0000,130,0.0000,Open,;
3,3,7662,7663,6.4606,250.0000,130,0.0000,Open,;
4,4,7661,7662,6.8711,250.0000,130,0.0000,Open,;
...,...,...,...,...,...,...,...,...,...
7661,361,169,2499,1.0000,32.0000,110,0.0000,Open,;
7662,634,2348,3736,1,200,137,0,Open,;
7663,672,5620,5619,1,90,135,0,Open,;
7664,10,6938,10000,1,300,135,0,Open,;


In [28]:
df_pipes['Length'] = df_pipes['Length'].astype(float)
df_pipes['Diameter'] = df_pipes['Diameter'].astype(float)

In [ ]:
# df_pipes.to_excel('Tabelas para calibração\\Projeto Inacio\\Saida\\Rede_EPANET Projeto.xlsx')

In [7]:
# diametros = df_pipes.groupby('Diameter')['Length'].sum().round(2).reset_index()
# diametros.sort_values(by='Diameter', ascending=True, inplace=True)
# diametros

# # calcula a soma
# soma_length = diametros['Length'].sum()

# # cria uma nova linha
# nova_linha = pd.DataFrame({'Diameter': ['Total'], 'Length': [soma_length]})

# # concatena ao DataFrame original
# diametros = pd.concat([diametros, nova_linha], ignore_index=True)

# diametros

# diametros.to_excel('Tabelas para calibração\\Projeto Inacio\\Saida\\Diametros EPANET_projeto.xlsx')

In [ ]:
# rugosidade = df_pipes.groupby('Roughness')['Length'].sum().round(2).reset_index()
# rugosidade

# rugosidade.to_excel('Tabelas para calibração\\Projeto Inacio\\Saida\\Rugosidade EPANET_projeto.xlsx')

In [8]:
# rede_inacio = pd.read_excel('Tabelas para calibração\\Projeto Inacio\\Entrada\\Rede.xls')
# rede_inacio.columns

In [9]:
# diametros_r = rede_inacio.groupby('diameter')['Shape__Len'].sum().round(2).reset_index()
# diametros_r.sort_values(by='diameter', ascending=True, inplace=True)
# diametros_r

# # calcula a soma
# soma_length = diametros_r['Shape__Len'].sum()

# # cria uma nova linha
# nova_linha = pd.DataFrame({'diameter': ['Total'], 'Shape__Len': [soma_length]})

# # concatena ao DataFrame original
# diametros_r = pd.concat([diametros_r, nova_linha], ignore_index=True)

# diametros_r

# diametros_r.to_excel('Tabelas para calibração\\Projeto Inacio\\Saida\\Diametros Cadastro Tecnico.xlsx')

In [10]:
# Material_r = (
#     rede_inacio
#     .groupby(['material','rugosidade'])['Shape__Len']
#     .sum()
#     .round(2)
#     .reset_index()
# )

# # ordenar pelo comprimento, ou outro critério que queira
# Material_r.sort_values(by='rugosidade', ascending=True, inplace=True)

# # soma do comprimento
# soma_length = Material_r['Shape__Len'].sum()

# # linha extra
# nova_linha = pd.DataFrame({
#     'rugosidade': ['Total'],
#     'material': ['-'],
#     'Shape__Len': [soma_length]
# })

# # concatenar
# Material_r = pd.concat([Material_r, nova_linha], ignore_index=True)

# Material_r.to_excel('Tabelas para calibração\\Projeto Inacio\\Saida\\Material e Rugosidade Cadastro Tecnico.xlsx')


In [62]:
df_pipes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8796 entries, 0 to 8795
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ID         8796 non-null   object
 1   Node1      8796 non-null   object
 2   Node2      8796 non-null   object
 3   Length     8796 non-null   object
 4   Diameter   8796 non-null   object
 5   Roughness  8796 non-null   object
 6   MinorLoss  8796 non-null   object
 7   Status     8796 non-null   object
 8   empty      8796 non-null   object
dtypes: object(9)
memory usage: 618.6+ KB


In [ ]:
# df_pipes.to_excel('Tabelas para calibração\\Paranoá-Itapoã\\rede atual PRN ITP.xlsx')

In [ ]:
# rede_calibra = pd.read_csv('Epanet\\Brazlândia\\uda1.csv')

In [11]:
# rede_calibra = pd.read_excel("Tabelas para calibração\\Brazlândia\\UDA.002\\Rede_calibração - parte superior antes dmc.xls")
# colunas_para_arredondar = ['Xinicial', 'Ynicial', 'Xfinal', 'Yfinal']  # Substitua pelos nomes corretos
# rede_calibra[colunas_para_arredondar] = rede_calibra[colunas_para_arredondar].round(3)
# rede_calibra

In [ ]:
# #Correspondencia com a tabela original através das coordenadas para encontrar o ID da rede que foi para o Epanet
# df_correspondencia = pd.merge(df_pipes_com_coords,rede_calibra, 
#                               left_on=['x_start', 'y_start', 'x_end', 'y_end'],
#                               right_on=['Xinicial', 'Ynicial', 'Xfinal', 'Yfinal'], 
#                               how='inner')

In [ ]:
# df_correspondencia

,ID,Node1,Node2,x_start,y_start,x_end,y_end,FID,ASSETGROUP,ASSETTYPE,...,rugosidade,dataimplan,Shape__Len,ORIG_FID,ORIG_SEQ,Xinicial,Ynicial,Xfinal,Yfinal,UDA
0,1336,2203,2204,156843.146,8264505.723,156843.138,8264499.723,2411,1,2,...,132.5,1994-02-22,6.000005,1609,1,156843.146,8264505.723,156843.138,8264499.723,UDA.BRZ.002
1,1337,2201,2203,156847.472,8264668.443,156843.146,8264505.723,2412,1,2,...,132.5,1994-02-22,162.776897,1610,1,156847.472,8264668.443,156843.146,8264505.723,UDA.BRZ.002
2,1340,459,1079,156846.423,8264670.612,156835.772,8264670.562,2433,1,2,...,135.0,1992-03-30,10.652015,1627,1,156846.423,8264670.612,156835.772,8264670.562,UDA.BRZ.002
3,1347,2202,459,156847.523,8264670.642,156846.423,8264670.612,2410,1,2,...,132.5,1994-02-22,1.100028,1608,1,156847.523,8264670.642,156846.423,8264670.612,UDA.BRZ.002
4,1348,2021,2011,157027.584,8264720.632,157027.481,8264719.225,2110,1,2,...,135.0,1992-03-30,1.411198,1398,1,157027.584,8264720.632,157027.481,8264719.225,UDA.BRZ.002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,2594,1091,1090,156812.251,8264310.715,156814.276,8264310.425,927,1,2,...,125.0,1974-12-12,2.045363,563,1,156812.251,8264310.715,156814.276,8264310.425,UDA.BRZ.002
164,2595,1092,1090,156814.287,8264308.927,156814.276,8264310.425,928,1,2,...,125.0,1974-12-12,1.498146,564,1,156814.287,8264308.927,156814.276,8264310.425,UDA.BRZ.002
165,2598,197,1076,156718.568,8264276.796,156758.368,8264271.616,915,1,2,...,115.0,1974-12-12,40.135675,551,1,156718.568,8264276.796,156758.368,8264271.616,UDA.BRZ.002
166,2783,1073,1748,156656.608,8264289.430,156642.176,8264282.085,2105,1,2,...,115.0,1974-12-12,16.193348,1393,1,156656.608,8264289.430,156642.176,8264282.085,UDA.BRZ.002


In [12]:
# df_correspondencia = pd.merge(df_pipes_com_coords,rede_calibra, 
#                               left_on=['x_start', 'y_start', 'x_end', 'y_end'],
#                               right_on=['Xinicial', 'Ynicial', 'Xfinal', 'Yfinal'], 
#                               how='inner')

In [ ]:
# lista_ids = df_correspondencia['ID'].astype(str).tolist()
lista_ids = rede_calibra['Pipe'].astype(str).tolist()



In [ ]:
# df_pipes["Roughness"] = df_pipes["Roughness"].astype(int)

In [14]:
# df_pipes[df_pipes['ID']=='3000']

In [ ]:
list_pipe_id = lista_ids
new_adjust_roughness = 5

df_pipes = adjust_roughness_with_limit(df_pipes, 
                            list_pipe_id, 
                            new_adjust_roughness, 
                            limit=40)

In [192]:
df_pipes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3116 entries, 0 to 3115
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ID         3116 non-null   object
 1   Node1      3116 non-null   object
 2   Node2      3116 non-null   object
 3   Length     3116 non-null   object
 4   Diameter   3116 non-null   object
 5   Roughness  3116 non-null   int32 
 6   MinorLoss  3116 non-null   object
 7   Status     3116 non-null   object
 8   empty      3116 non-null   object
dtypes: int32(1), object(8)
memory usage: 207.1+ KB


In [336]:
"3000" in lista_ids

False

In [361]:
df_pipes[df_pipes['ID']=='3000']

,ID,Node1,Node2,Length,Diameter,Roughness,MinorLoss,Status,empty
2898,3000,1984,1985,22.247336,85,167,0,Open,;


In [362]:
df_pipes

,ID,Node1,Node2,Length,Diameter,Roughness,MinorLoss,Status,empty
0,0,1272,1266,41.412884,60,132,0,Open,;
1,1,509,2568,4.608553,60,132,0,Open,;
2,2,2566,2567,3.41083,60,132,0,Open,;
3,4,2754,2640,75.732263,50,85,0,Open,;
4,6,1813,2734,34.226609,150,166,0,Open,;
...,...,...,...,...,...,...,...,...,...
3111,155,564,2251,8.8490,60,135,0,Open,;
3112,156,564,2252,8.1959,60,135,0,Open,;
3113,215,835,976,38.81,60,127,0,Open,;
3114,216,2014,2015,3.95,110,132,0,Open,;


In [151]:
# list_pipe_id=[0, 1, 2, 3]
# new_roughness = 98765
# df_pipes = setLinkRoughnessCoeff(df_pipes, list_pipe_id=list_pipe_id, 
#                                  new_roughness=new_roughness)

In [29]:
df_pipes.loc[df_pipes['Diameter'] == 32, ['Diameter', 'Roughness']] = [63, 140]


In [30]:
df_pipes['Diameter'].unique()

array([110., 250.,  63., 200.,  90., 160.,  60., 225.,  85., 180., 255.,
       125., 400., 500., 280., 315.,  75., 100., 150.,  50., 300.])

In [31]:
df_pipes

,ID,Node1,Node2,Length,Diameter,Roughness,MinorLoss,Status,empty
0,0,7867,7504,3.2711,110.0,140.0000,0.0000,Open,;
1,1,7664,7665,4.5834,250.0,130,0.0000,Open,;
2,2,7663,7664,11.8994,250.0,130,0.0000,Open,;
3,3,7662,7663,6.4606,250.0,130,0.0000,Open,;
4,4,7661,7662,6.8711,250.0,130,0.0000,Open,;
...,...,...,...,...,...,...,...,...,...
7661,361,169,2499,1.0000,63.0,140,0.0000,Open,;
7662,634,2348,3736,1.0000,200.0,137,0,Open,;
7663,672,5620,5619,1.0000,90.0,135,0,Open,;
7664,10,6938,10000,1.0000,300.0,135,0,Open,;


In [32]:
new_arquivo = "Epanet\\Sao Sebastiao\\Novo plano\\SSB - SO 63.inp"
update_pipes_inp(df_pipes=df_pipes, arquivo_inp=new_arquivo)

In [368]:
nos = pd.read_excel('Tabelas para calibração\\Brazlândia\\Demanda_BRZ.xlsx')
nos.columns

Index(['Unnamed: 0', 'Input_FID', 'Demanda', 'perda', 'Demanda+perda'], dtype='object')

In [369]:
thiessen = pd.read_excel("Tabelas para calibração\\Brazlândia\\Thiessen_conserto.xls")
thiessen.columns

Index(['OBJECTID', 'Join_Count', 'TARGET_FID', 'Id', 'Input_FID', 'ASSETGROUP',
       'ASSETTYPE', 'FROMDEVICE', 'TODEVICETE', 'GLOBALID', 'creationda',
       'creator', 'lastupdate', 'updatedby', 'installdat', 'notes', 'diameter',
       'lifecycles', 'inserviced', 'retireddat', 'material', 'designtype',
       'codunidade', 'posicionam', 'contratoob', 'tiposistem', 'tipodesenh',
       'rugosidade', 'dataimplan', 'Shape__Len', 'ORIG_FID', 'ORIG_SEQ',
       'Xinicial', 'Ynicial', 'Xfinal', 'Yfinal', 'UDA', 'Cota', 'POINT_X',
       'POINT_Y', 'POINT_Z', 'POINT_M', 'Data_da_ab', 'Localidade',
       'F_OS__UDAs', 'DMC', 'Tipo_de_se', 'Código_e_', 'OS', 'LATITUDE',
       'LONGITUDE', 'Shape_Length', 'Shape_Area'],
      dtype='object')

In [382]:
unido = pd.merge(nos,thiessen,on="Input_FID",how="right")
unido

,Unnamed: 0,Input_FID,Demanda,perda,Demanda+perda,OBJECTID,Join_Count,TARGET_FID,Id,ASSETGROUP,...,Localidade,F_OS__UDAs,DMC,Tipo_de_se,Código_e_,OS,LATITUDE,LONGITUDE,Shape_Length,Shape_Area
0,1209.0,1970,0.037249,0.007195,0.044444,1,1,0,0,1,...,Brazlandia,UDA.BRZ.002,DMC.BRZ.022,CAVALETE,(8101008011055) - Conserto de cavalete e regis...,2.725700e+15,-15.689928,-48.204588,1097.627683,64301.587445
1,NaN,1969,NaN,NaN,NaN,2,0,1,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,767.091174,14666.780331
2,NaN,1964,NaN,NaN,NaN,3,0,2,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1001.579754,12275.007272
3,1208.0,1968,0.037797,0.010792,0.048589,4,0,3,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,890.741097,8311.372642
4,1205.0,1963,0.061130,0.003597,0.064728,5,0,4,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1025.428526,9052.815342
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2373,NaN,2092,NaN,NaN,NaN,2374,0,2373,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,526.138275,12309.809279
2374,NaN,2100,NaN,NaN,NaN,2375,0,2374,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,433.867444,7455.713581
2375,NaN,1626,NaN,NaN,NaN,2376,0,2375,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,427.240647,4162.413887
2376,1051.0,1621,0.066721,0.021133,0.087854,2377,3,2376,0,1,...,Brazlandia,UDA.BRZ.003,Não informada,CAVALETE,(8101008011055) - Conserto de cavalete e regis...,2.725585e+15,-15.653760,-48.195978,252.025219,3822.250166


In [384]:
unido = unido[["Input_FID","Demanda", "perda","Join_Count","UDA"]]
unido['Demanda'].fillna(0,inplace=True)
unido['perda'].fillna(0,inplace=True)
unido

C:\Users\filipe_silva\AppData\Local\Temp\ipykernel_46556\561620297.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  unido['Demanda'].fillna(0,inplace=True)
C:\Users\filipe_silva\AppData\Local\Temp\ipykernel_46556\561620297.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For 

,Input_FID,Demanda,perda,Join_Count,UDA
0,1970,0.037249,0.007195,1,UDA.BRZ.002
1,1969,0.000000,0.000000,0,UDA.BRZ.002
2,1964,0.000000,0.000000,0,UDA.BRZ.002
3,1968,0.037797,0.010792,0,UDA.BRZ.002
4,1963,0.061130,0.003597,0,UDA.BRZ.002
...,...,...,...,...,...
2373,2092,0.000000,0.000000,0,UDA.BRZ.003
2374,2100,0.000000,0.000000,0,UDA.BRZ.003
2375,1626,0.000000,0.000000,0,UDA.BRZ.003
2376,1621,0.066721,0.021133,3,UDA.BRZ.003


In [389]:
df1 = unido[unido['UDA']=='UDA.BRZ.001']
df2 = unido[unido['UDA']=='UDA.BRZ.002']
df3 = unido[unido['UDA']=='UDA.BRZ.003']
df1['perda'].sum()

13.464994521575003

In [393]:
df1 = df1.rename(columns={"Input_FID": "NODENUM"})
df2 = df2.rename(columns={"Input_FID": "NODENUM"})
df3 = df3.rename(columns={"Input_FID": "NODENUM"})

In [396]:
import numpy as np
import pandas as pd

def ajustar_perdas_com_pesos(perda_total, dic_df):
    # Agrupar os dados por nó e somar as perdas e o número de serviços
    perdas_por_no = dic_df.groupby('NODENUM')['perda'].sum().to_dict()
    servicos_por_no = dic_df.groupby('NODENUM')['Join_Count'].sum().to_dict()

    # Garantir que não há serviços negativos
    servicos_por_no = np.array([servicos_por_no.get(no, 0) for no in perdas_por_no.keys()])
    
    # Criar pesos adaptativos com base na quantidade de serviços
    min_servicos = np.min(servicos_por_no)
    max_servicos = np.max(servicos_por_no)

    if min_servicos == max_servicos:
        # Se todos os nós têm a mesma quantidade de serviços, distribuir uniformemente
        pesos = np.ones_like(servicos_por_no)
    else:
        # Normalizar pesos entre 0.5 e 1.5 proporcionalmente aos serviços
        pesos = (servicos_por_no - min_servicos) / (max_servicos - min_servicos)

    # Ajustar as perdas para cada nó com base nas ligações
    perdas_ajustadas = np.zeros_like(servicos_por_no, dtype=float)

    for i, no in enumerate(perdas_por_no.keys()):
        perdas_ajustadas[i] = perdas_por_no[no] * pesos[i]

    # Normalizar as perdas para não ultrapassar a perda total
    perdas_ajustadas = perdas_ajustadas * (perda_total / np.sum(perdas_ajustadas))

    # Criar um dicionário para associar as perdas ajustadas aos nós
    perdas_ajustadas_dict = dict(zip(perdas_por_no.keys(), perdas_ajustadas))

    # Adicionar a coluna 'perda_ajustada' no DataFrame
    dic_df['perda_ajustada'] = dic_df['NODENUM'].map(perdas_ajustadas_dict)

    return dic_df


# # df = pd.DataFrame(data)

# perda_total = unido['perda'].sum()
# df_com_perda_ajustada = ajustar_perdas_com_pesos(perda_total, unido)

In [397]:
dic_df = {"df1": df1, "df2": df2,"df3": df3}

In [398]:
# df = pd.DataFrame(data)

perda_total = df1['perda'].sum()
df_com_perda_ajustada = ajustar_perdas_com_pesos(perda_total, df1)

In [399]:
# df = pd.DataFrame(data)

perda_total = df2['perda'].sum()
df_com_perda_ajustada = ajustar_perdas_com_pesos(perda_total, df2)

In [400]:
# df = pd.DataFrame(data)

perda_total = df3['perda'].sum()
df_com_perda_ajustada = ajustar_perdas_com_pesos(perda_total, df3)

In [409]:
# df2['perda_ajustada'].sum()
df2['perda'].sum()

16.752752726654997

In [403]:
perda_ajustada = pd.concat([df1,df2,df3])
perda_ajustada

,NODENUM,Demanda,perda,Join_Count,UDA,perda_ajustada
289,1234,0.018144,0.009427,0,UDA.BRZ.001,0.000000
462,28,0.036520,0.028281,3,UDA.BRZ.001,0.067375
464,1117,0.062388,0.050278,2,UDA.BRZ.001,0.079852
468,1135,0.015783,0.006285,1,UDA.BRZ.001,0.004991
469,810,0.053287,0.021996,0,UDA.BRZ.001,0.000000
...,...,...,...,...,...,...
2373,2092,0.000000,0.000000,0,UDA.BRZ.003,0.000000
2374,2100,0.000000,0.000000,0,UDA.BRZ.003,0.000000
2375,1626,0.000000,0.000000,0,UDA.BRZ.003,0.000000
2376,1621,0.066721,0.021133,3,UDA.BRZ.003,0.051997


In [410]:
perda_ajustada['demanda_final']= perda_ajustada['Demanda']+perda_ajustada['perda_ajustada']

In [426]:
perda_ajustada=perda_ajustada[['NODENUM','demanda_final']]
perda_ajustada = perda_ajustada.sort_values(by='NODENUM', ascending=True)
perda_ajustada = perda_ajustada.rename(columns={"NODENUM": "ID"})

In [440]:
lista_nos = perda_ajustada[["ID", "demanda_final"]]#.itertuples(index=False, name=None)
lista_nos #= list(lista_nos)


,ID,demanda_final
93,0,0.000000
91,1,0.000000
146,2,0.000000
102,3,0.000000
220,4,0.000000
...,...,...
61,2738,0.000000
15,2739,0.000000
562,2758,0.007102
65,2768,0.006022


# Para nós

In [ ]:
# import pandas as pd

# def extrair_junctions(arquivo_inp):
#     """Extrai os dados da seção [JUNCTIONS] do arquivo .inp e retorna um DataFrame"""
#     # with open(arquivo_inp, 'r', encoding='utf-8') as f:
#     with open(arquivo_inp, 'r', encoding='latin1') as f:
#         linhas = f.readlines()

#     junctions_data = []
#     coletando = False  # Flag para indicar quando estamos na seção [JUNCTIONS]

#     for linha in linhas:
#         linha = linha.strip()
#         if linha.startswith("[JUNCTIONS]"):
#             coletando = True
#             continue
#         if coletando:
#             if linha == "" or linha.startswith("["):  # Sai da seção se encontrar outra seção
#                 break
#             if not linha.startswith(";"):  # Ignora comentários
#                 parts = linha.split()
#                 if len(parts) >= 3:  # Garante que tem pelo menos ID, Elevação e Demanda
#                     id_no = parts[0]
#                     elevacao = float(parts[1])
#                     demanda = float(parts[2])
#                     padrao = parts[3] if len(parts) > 3 else ""
#                     junctions_data.append([id_no, elevacao, demanda, padrao])

#     colunas = ["ID", "Elev", "Demand", "Pattern"]
#     df_junctions = pd.DataFrame(junctions_data, columns=colunas)

#     return df_junctions



# def atualizar_junctions(arquivo_inp, df_junctions):
#     """Atualiza a seção [JUNCTIONS] no arquivo .inp com os novos valores de demanda"""
#     with open(arquivo_inp, 'r', encoding='utf-8') as f:
#         linhas = f.readlines()

#     nova_secao_junctions = []
#     coletando = False

#     for linha in linhas:
#         linha_strip = linha.strip()

#         if linha_strip.startswith("[JUNCTIONS]"):
#             coletando = True
#             nova_secao_junctions.append("[JUNCTIONS]\n")
#             nova_secao_junctions.append(";ID              	Elev        	Demand      	Pattern\n")
#             continue
#         elif coletando and (linha_strip == "" or linha_strip.startswith("[")):  
#             coletando = False

#             # Adiciona os junctions atualizados
#             for _, row in df_junctions.iterrows():
#                 nova_linha = f"{row['ID']: <18}\t{row['Elev']: <12.5f}\t{row['Demand']: <12.6f}\t{row['Pattern']: <12}\n"
#                 nova_secao_junctions.append(nova_linha)
        
#         if not coletando:
#             nova_secao_junctions.append(linha)

#     # Escrevendo de volta no arquivo .inp
#     with open(arquivo_inp, 'w', encoding='utf-8') as f:
#         f.writelines(nova_secao_junctions)

# def alterar_demandas(df_junctions, demandas_dict):
#     """
#     Atualiza a demanda dos nós com base em um dicionário {ID: nova_demanda}.
    
#     Parâmetros:
#         df_junctions (DataFrame): DataFrame dos junctions.
#         demandas_dict (dict): Dicionário contendo {ID: nova_demanda}.
    
#     Retorna:
#         DataFrame atualizado.
#     """
#     for id_no, nova_demanda in demandas_dict.items():
#         df_junctions.loc[df_junctions["ID"] == str(id_no), "Demand"] = nova_demanda

#     return df_junctions


In [28]:
import pandas as pd

# ==========================
# EXTRAIR JUNCTIONS
# ==========================
def extrair_junctions(arquivo_inp):
    """Extrai os dados da seção [JUNCTIONS] do arquivo .inp e retorna um DataFrame"""

    with open(arquivo_inp, 'r', encoding='latin1') as f:
        linhas = f.readlines()

    junctions_data = []
    coletando = False

    for linha in linhas:
        linha = linha.strip()

        if linha.upper().startswith("[JUNCTIONS]"):
            coletando = True
            continue

        if coletando:

            # Sai só quando começa outra seção
            if linha.startswith("[") and not linha.upper().startswith("[JUNCTIONS]"):
                break

            if linha == "":
                continue

            # 🔥 SEPARA DADOS E DESCRIÇÃO
            if ";" in linha:
                parte_dados, parte_desc = linha.split(";", 1)
                descricao = parte_desc.strip()
            else:
                parte_dados = linha
                descricao = ""

            if parte_dados.strip() == "":
                continue

            parts = parte_dados.split()

            if len(parts) >= 2:
                try:
                    id_no = parts[0]
                    elevacao = float(parts[1])
                    demanda = float(parts[2]) if len(parts) > 2 else 0.0
                    padrao = parts[3] if len(parts) > 3 else ""

                    junctions_data.append([
                        id_no, elevacao, demanda, padrao, descricao
                    ])

                except Exception as e:
                    print(f"Erro na linha: {linha} | {e}")

    colunas = ["ID", "Elev", "Demand", "Pattern", "Descricao"]
    df_junctions = pd.DataFrame(junctions_data, columns=colunas)

    print(f"📊 Total de nós carregados: {len(df_junctions)}")

    return df_junctions


# ==========================
# ATUALIZAR JUNCTIONS NO INP
# ==========================
def atualizar_junctions(arquivo_inp, df_junctions):
    """Atualiza a seção [JUNCTIONS] no arquivo .inp com os novos valores"""

    with open(arquivo_inp, 'r', encoding='latin1') as f:
        linhas = f.readlines()

    nova_secao = []
    coletando = False

    for linha in linhas:
        linha_strip = linha.strip()

        # INÍCIO DA SEÇÃO
        if linha_strip.upper().startswith("[JUNCTIONS]"):
            coletando = True
            nova_secao.append("[JUNCTIONS]\n")
            nova_secao.append(";ID               Elev         Demand       Pattern        ;Descricao\n")
            continue

        # FIM DA SEÇÃO
        elif coletando and (linha_strip.startswith("[") and not linha_strip.upper().startswith("[JUNCTIONS]")):
            coletando = False

            # 🔥 ESCREVE OS DADOS ATUALIZADOS
            for _, row in df_junctions.iterrows():

                descricao = f";{row['Descricao']}" if row['Descricao'] else ""

                nova_linha = (
                    f"{str(row['ID']): <18}"
                    f"{row['Elev']: <12.5f}"
                    f"{row['Demand']: <12.6f}"
                    f"{str(row['Pattern']): <14}"
                    f"{descricao}\n"
                )

                nova_secao.append(nova_linha)

        # FORA DA SEÇÃO → mantém original
        if not coletando:
            nova_secao.append(linha)

    # SALVA
    with open(arquivo_inp, 'w', encoding='latin1') as f:
        f.writelines(nova_secao)

    print("✅ Arquivo .inp atualizado com sucesso!")


# ==========================
# ALTERAR DEMANDAS
# ==========================
def alterar_demandas(df_junctions, demandas_dict):
    """
    Atualiza a demanda dos nós com base em um dicionário {ID: nova_demanda}.
    """

    for id_no, nova_demanda in demandas_dict.items():
        df_junctions.loc[
            df_junctions["ID"] == str(id_no),
            "Demand"
        ] = nova_demanda

    return df_junctions

In [29]:
# Caminho do arquivo
arquivo = r"Epanet\Gama 2\GAM2 V20 ajustada.inp"

# Criar DataFrame de Junctions
df_junctions = extrair_junctions(arquivo)
df_junctions

📊 Total de nós carregados: 7862


,ID,Elev,Demand,Pattern,Descricao
0,0,1098.87,5.181450,VZ1.DMC.GAM.003,
1,1,1098.87,0.000000,VZ1.DMC.GAM.003,
2,2,1088.07,0.004887,VZ1.DMC.GAM.004,
3,3,1087.66,0.017747,VZ1.DMC.GAM.004,
4,4,1088.41,0.000000,VZ1.DMC.GAM.004,
...,...,...,...,...,...
7857,185,1212.00,0.000000,,
7858,186,1212.00,0.000000,,
7859,495,1212.50,0.000000,,
7860,496,1217.00,0.000000,,


In [39]:
df_junctions['Pattern'].unique()

array(['VZ1.DMC.SAM.002', 'VZ1.RAP.SAM.002', 'VZ1.DMC.SAM.001',
       'VZ1.DMC.SAM.004', 'VZ1.AAT.SAM.030', 'VZ1.DMC.SAM.003', ''],
      dtype=object)

In [15]:
# Exemplo: supondo que seu DataFrame se chame df
df_junctions.loc[df_junctions['Pattern'].str.contains('VZ1.DMC.RCE.001', case=False, na=False), 'Pattern'] = 'VZ1.AAT.RCE.010'

In [66]:
# Top 10 maiores valores de Demand com o respectivo ID
top10 = df_junctions.nlargest(10, "Demand")[["ID", "Demand"]]

print(top10)


          ID    Demand
3681    3698  4.295992
1870    1884  1.872410
1883    1897  1.242548
8353    8495  1.238150
15553  16622  1.233245
15565  16634  1.190531
1361    1361  1.067084
8411    8553  1.035385
893      893  0.988116
6713    6739  0.980765


In [9]:
df_junctions.to_excel('Epanet\\Projeto Inacio\\Parte de cima.xlsx')

In [ ]:
# demanda = pd.read_excel('Tabelas para calibração\\Paranoá-Itapoã\\Nós Smlin remover.xls')
# demanda.columns
# demanda = demanda[['Ligações.INSCRICAO_','BDGIS.TB_GCOMConsumo12Meses.TipoAgrupamento','Ligações.RADESC','BDGIS.TB_GCOMConsumo12Meses.MediaVolumeConsumido']]

Index(['FID', 'Join_Count', 'TARGET_FID', 'Join_Cou_1', 'TARGET_F_1',
       'ASSETGROUP', 'ASSETTYPE', 'FROMDEVICE', 'TODEVICETE', 'GLOBALID',
       'creationda', 'creator', 'lastupdate', 'updatedby', 'installdat',
       'notes', 'diameter', 'lifecycles', 'inserviced', 'retireddat',
       'material', 'designtype', 'codunidade', 'posicionam', 'contratoob',
       'tiposistem', 'tipodesenh', 'rugosidade', 'dataimplan', 'Shape__Len',
       'ORIG_FID', 'ORIG_SEQ', 'Xinicial', 'Yinicial', 'Xfinal', 'Yfinal',
       'Sistema', 'Localidade', 'RAP', 'UDA', 'DMC', 'BOOSTER', 'VRP',
       'created_us', 'created_da', 'last_edite', 'last_edi_1', 'TAG_VAZAO',
       'Zonapressa', 'ZonaManobr', 'GerenciaMa', 'OBJECTID_1', 'NOME',
       'SHAPE_Leng', 'REGIÃO', 'RA', 'Shape_Le_1', 'Shape_Area', 'Cota',
       'POINT_X', 'POINT_Y', 'POINT_Z', 'POINT_M'],
      dtype='object')

In [105]:
# demanda

In [ ]:
# demanda['FID'].value_counts().sum()

196

In [30]:
# Nos = pd.read_excel('Tabelas para calibração\\Samambaia\\Nos_parciais_SAM.xlsx')
Nos = pd.read_excel('Tabelas para calibração\\Gama 2\\Nos_parciais GAM2.xlsx')

In [33]:
Nos

,Unnamed: 0,NODENUM,Consumo_Original,Consumo CANF,Consumo,Join_Count_x,perda,perda_ajustada,Demanda,Join_Count_y,...,ZonaManobr,GerenciaMa,Validado,COTA,POINT_X,POINT_Y,POINT_Z,POINT_M,Longitude,Latitude
0,0,0,0.000000,7.36,7.360000,0.0,0.024679,0.0,7.360000,1,...,,PASS,sim,1098.869995,170226.0692,8.226404e+06,0.0,NaN,-48.081281,-16.019992
1,1,1,0.000000,0.00,0.000000,0.0,0.024679,0.0,0.000000,1,...,,PASS,sim,1098.869995,170227.1863,8.226404e+06,0.0,NaN,-48.081270,-16.019999
2,2,2,0.004887,0.00,0.004887,0.0,0.024679,0.0,0.004887,1,...,,PASS,sim,1088.069946,172359.2492,8.224123e+06,0.0,NaN,-48.061684,-16.040869
3,3,3,0.017747,0.00,0.017747,0.0,0.024679,0.0,0.017747,1,...,,PASS,sim,1087.660034,172380.2812,8.224111e+06,0.0,NaN,-48.061489,-16.040984
4,4,4,0.000000,0.00,0.000000,0.0,0.024679,0.0,0.000000,1,...,,PASS,sim,1088.410034,172336.4595,8.224138e+06,0.0,NaN,-48.061895,-16.040730
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8130,8588,8588,0.001929,0.00,0.001929,0.0,0.024679,0.0,0.001929,1,...,,PASS,sim,1112.359985,173274.8959,8.225515e+06,0.0,NaN,-48.052945,-16.028429
8131,8589,8589,0.000000,0.00,0.000000,0.0,0.024679,0.0,0.000000,1,...,,PASS,sim,1112.359985,173272.1688,8.225517e+06,0.0,NaN,-48.052970,-16.028413
8132,8590,8590,0.000000,0.00,0.000000,0.0,0.024679,0.0,0.000000,1,...,,PASS,sim,1112.359985,173273.0950,8.225516e+06,0.0,NaN,-48.052962,-16.028419
8133,8591,8591,0.000000,0.00,0.000000,0.0,0.024679,0.0,0.000000,1,...,,PASS,sim,1112.359985,173273.7017,8.225516e+06,0.0,NaN,-48.052956,-16.028422


In [34]:
Nos['UDA'].value_counts()

UDA
UDA.GAM.001    4386
UDA.GAM.002    2198
UDA.GAM.003     889
UDA.GAM.004     473
                189
Name: count, dtype: int64

In [6]:
df_junctions

,ID,Elev,Demand,Pattern,Descricao
0,0,1055.21,0.0,VZ1.RAP.SAM.002,
1,1,1055.06,0.0,VZ1.DMC.SAM.002,
2,2,1156.50,0.0,VZ1.DMC.SAM.002,
3,3,1156.50,0.0,VZ1.RAP.SAM.002,
4,4,1156.50,0.0,VZ1.RAP.SAM.002,
...,...,...,...,...,...
21477,21496,1239.70,0.0,VZ1.AAT.SAM.050,
21478,21497,1239.70,0.0,VZ1.AAT.SAM.050,
21479,PM.VRP.SAM.017,1090.00,0.0,VZ1.RAP.SAM.002,
21480,488,1182.60,0.0,VZ1.RAP.SAM.002,Fechado pela ZP - cadastrar VMA


In [9]:
apoio = pd.read_excel('Tabelas para calibração\\Samambaia\\Tabela_correta_epanet.xlsx')

In [10]:
apoio.columns

Index(['FID', 'Join_Count', 'TARGET_FID', 'ID', 'ELEVATION', 'DEMAND',
       'Sistema', 'Localidade', 'RAP', 'UDA', 'DMC', 'BOOSTER', 'VRP',
       'created_us', 'created_da', 'last_edite', 'last_edi_1', 'TAG_VAZAO',
       'Zonapressa', 'ZonaManobr', 'GerenciaMa', 'Validado'],
      dtype='object')

In [35]:
# Nos['NODENUM'] = Nos['NODENUM'].astype(int)
# df_junctions['ID'] = df_junctions['ID'].astype(int)

# Garantir que as colunas estejam como string
Nos['NODENUM'] = Nos['NODENUM'].astype(str)
df_junctions['ID'] = df_junctions['ID'].astype(str)

In [11]:
df_junctions['ID'] = df_junctions['ID'].astype(str)
apoio['ID'] = apoio['ID'].astype(str)

In [26]:
import pandas as pd

# 1️⃣ Filtrar o DataFrame 'Nos' para pegar todos com UDA = 'UDA.RCE.002'
lista_nos = Nos.loc[Nos['UDA'] == 'UDA.RCE.002', 'NODENUM'].tolist()

# 2️⃣ Atualizar o DataFrame 'df_juctions'
df_junctions.loc[df_junctions['ID'].isin(lista_nos), 'Pattern'] = 'VZ1.SAT.RCE.031'


In [40]:
# Garantir que as colunas estejam como string
Nos['NODENUM'] = Nos['NODENUM'].astype(str)
df_junctions['ID'] = df_junctions['ID'].astype(str)

# Filtrar nós da UDA desejada
lista_nos = Nos.loc[Nos['UDA'] == 'UDA.RCE.002', 'NODENUM'].tolist()

# Atualizar o pattern no df_juctions com base nos IDs encontrados
df_junctions.loc[df_junctions['ID'].isin(lista_nos), 'Pattern'] = 'VZ1.SAT.RCE.031'

# Conferir resultado
print(df_junctions.loc[df_junctions['ID'].isin(lista_nos), ['ID', 'Pattern']].head(10))
print(f"Total de linhas atualizadas: {df_junctions['Pattern'].eq('VZ1.SAT.RCE.031').sum()}")


  ID          Pattern
0  0  VZ1.SAT.RCE.031
1  1  VZ1.SAT.RCE.031
2  2  VZ1.SAT.RCE.031
3  3  VZ1.SAT.RCE.031
4  4  VZ1.SAT.RCE.031
5  5  VZ1.SAT.RCE.031
6  6  VZ1.SAT.RCE.031
7  7  VZ1.SAT.RCE.031
8  8  VZ1.SAT.RCE.031
9  9  VZ1.SAT.RCE.031
Total de linhas atualizadas: 4113


In [36]:
df_junctions.loc[df_junctions['ID'].isin(lista_nos), 'Pattern'] = 'VZ1.SAT.RCE.031'


In [41]:
df_junctions.loc[df_junctions['ID'].isin(lista_nos), ['ID', 'Pattern']].head(10)


,ID,Pattern
0,0,VZ1.SAT.RCE.031
1,1,VZ1.SAT.RCE.031
2,2,VZ1.SAT.RCE.031
3,3,VZ1.SAT.RCE.031
4,4,VZ1.SAT.RCE.031
5,5,VZ1.SAT.RCE.031
6,6,VZ1.SAT.RCE.031
7,7,VZ1.SAT.RCE.031
8,8,VZ1.SAT.RCE.031
9,9,VZ1.SAT.RCE.031


In [31]:
# Filtrar VRP desejados
filtro_vrp = Nos["VRP"].isin(["VRP.RCE.002", "VRP.RCE.004"])

# Excluir DMC indesejados
filtro_dmc = ~Nos["DMC"].isin(["DMC.RCE.001", "DMC.RCE.002"])

# Aplicar os dois filtros
resultado = Nos[filtro_vrp & filtro_dmc]["NODENUM"].tolist()

print(resultado)


[23, 24, 25, 26, 37, 38, 64, 65, 308, 309, 310, 311, 312, 313, 314, 315, 316, 319, 320, 321, 328, 329, 330, 331, 332, 333, 334, 355, 356, 369, 370, 371, 372, 373, 374, 375, 733, 734, 735, 736, 739, 740, 741, 742, 743, 744, 745, 746, 819, 820, 821, 827, 828, 831, 832, 833, 834, 851, 852, 853, 861, 862, 863, 864, 865, 866, 867, 868, 869, 870, 871, 1004, 1005, 1006, 1007, 1008, 1009, 1130, 1131, 1511, 1512, 1513, 1514, 1515, 1516, 2101, 2102, 2103, 2104, 2105, 2106, 2107, 2108, 2109, 2110, 2111, 2112, 2113, 2114, 2115, 2116, 2117, 2118, 2119, 2120, 2121, 2122, 2123, 2124, 2125, 2126, 2127, 2128, 2175, 2176, 2177, 2178, 2179, 2180, 2181, 2182, 2183, 2184, 2185, 2196, 2197, 2198, 2199, 2200, 2201, 2216, 2217, 2218, 2219, 2220, 2221, 2222, 2223, 2224, 2225, 2306, 2307, 2353, 2354, 2355, 2356, 2376, 2377, 2378, 2379, 2380, 2381, 2382, 2639, 2658, 2659, 2688, 2689, 2725, 2745, 2746, 2747, 2790, 2791, 2792, 2793, 2794, 2795, 2796, 2797, 2882, 2884, 2885, 2899, 2900, 2927, 2928, 2929, 2930, 2931

In [5]:
# id_para_tirar = demanda['FID'].astype(str)
# testando = Nos[Nos['NODENUM'].astype(str).isin(id_para_tirar)]
# # testando['Demand'].sum()
# testando

In [ ]:
# demanda['Ligações.RADESC'].value_counts()

Ligações.RADESC
Lago Sul           3263
Jardim Botanico      47
Name: count, dtype: int64

In [ ]:
# demanda = demanda[demanda['Ligações.RADESC'] != 'Jardim Botanico']

# demanda['Ligações.RADESC'].value_counts()

Ligações.RADESC
Lago Sul    3263
Name: count, dtype: int64

In [ ]:
# demanda['Consumo']= demanda['BDGIS.TB_GCOMConsumo12Meses.MediaVolumeConsumido']*1000/(30*24*60*60)
# demanda['Consumo'].sum()

C:\Users\filipe_silva\AppData\Local\Temp\ipykernel_33560\1542844673.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  demanda['Consumo']= demanda['BDGIS.TB_GCOMConsumo12Meses.MediaVolumeConsumido']*1000/(30*24*60*60)


30.908618827160495

In [27]:
df_junctions['Demand'].sum()

472.621032

In [ ]:
# Nos = pd.read_excel('Tabelas para calibração\\Paranoá-Itapoã\\Nós Tratado (Entre Lagos arrumado).xlsx')
# Nos.columns
# no_entre_lago = Nos[Nos['NODENUM']==3616]
# no_entre_lago

Index(['Unnamed: 0.1', 'Unnamed: 0', 'Input_FID_x', 'Consumo', 'NODENUM',
       'Join_Count_x', 'codunidade', 'Sistema', 'Localidade', 'RAP', 'UDA',
       'DMC', 'BOOSTER', 'VRP', 'TAG_VAZAO', 'Zonapressa', 'ZonaManobr',
       'GerenciaMa', 'REGIÃO', 'RA', 'Cota', 'POINT_X', 'POINT_Y', 'POINT_Z',
       'POINT_M', 'FID', 'DMC_1', 'VRP_1', 'Input_FID_y', 'Join_Count',
       'perda', 'perda_ajustada', 'Demanda', 'TAG_VAZAO_lista',
       'DMC_extraido'],
      dtype='object')

In [109]:
Nos[['RA']].value_counts()

RA     
PARANOÁ    5796
ITAPOÃ     1865
SMLIN       562
            111
Name: count, dtype: int64

In [7]:
Nos[['DMC','RAP']].value_counts()

DMC          RAP                    
             RAP.RCE.001                6898
             REL.RCE.001                5982
             REQ.GAM.001/REQ.GAM.002    2023
                                         944
DMC.RCE.002  RAP.RCE.001                 857
             RAP.RF2.001                 202
DMC.RCE.001  RAP.RCE.001                 178
Name: count, dtype: int64

In [ ]:
# Nos_arrumar_dmc04 = pd.read_excel('Tabelas para calibração\\Paranoá-Itapoã\\Nós para corrigir o DMC04-05.xls')
# ID_dmc4 = Nos_arrumar_dmc04['FID'].to_list()

In [ ]:
# # aplica a regra
# Nos.loc[
#     (Nos['DMC_1'] == 'PRN - DMC 4/5') & (Nos['NODENUM'].isin(ID_dmc4)),
#     'DMC_1'
# ] = 'PRN - DMC 4'

# Nos.loc[
#     (Nos['DMC_1'] == 'PRN - DMC 4/5') & (~Nos['NODENUM'].isin(ID_dmc4)),
#     'DMC_1'
# ] = 'PRN - DMC 5'

In [6]:
# Nos[['DMC_1','RAP']].value_counts()

In [20]:
Nos.columns

Index(['Unnamed: 0', 'NODENUM', 'Consumo_Original', 'Consumo CANF', 'Consumo',
       'Join_Count_x', 'perda', 'perda_ajustada', 'Demanda', 'Join_Count_y',
       'TARGET_FID', 'Join_Cou_1', 'TARGET_F_1', 'ASSETGROUP', 'ASSETTYPE',
       'ASSOCIATIO', 'ISCONNECTE', 'FROMDEVICE', 'TODEVICETE', 'GLOBALID',
       'cpsubnetwo', 'SUPPORTEDS', 'SUPPORTING', 'SystemSubn', 'PressureSu',
       'IsolationS', 'DMASubnetw', 'creationda', 'creator', 'lastupdate',
       'updatedby', 'installdat', 'notes', 'diameter', 'lifecycles',
       'inserviced', 'retireddat', 'material', 'designtype', 'codunidade',
       'posicionam', 'contratoob', 'tiposistem', 'tipodesenh', 'rugosidade',
       'dataimplan', 'FLOWDIRECT', 'trechoid', 'loteid', 'Shape__Len',
       'ORIG_FID', 'ORIG_SEQ', 'Xinicial', 'Xfinal', 'Yinicial', 'Yfinal',
       'Sistema', 'Localidade', 'RAP', 'UDA', 'DMC', 'BOOSTER', 'VRP',
       'created_us', 'created_da', 'last_edite', 'last_edi_1', 'TAG_VAZAO',
       'Zonapressa', 'ZonaMa

In [7]:
Nos['Zonapressa'].unique()

array(['VRP.SAM.013', 'VRP.SAM.006', 'VRP.SAM.007', 'VRP.SAM.011',
       'VRP.SAM.020', 'VRP.SAM.012', ' ', 'RAP.SAM.001', 'VRP.SAM.005',
       'VRP.SAM.019', 'VRP.SAM.016', 'VRP.SAM.002', 'VRP.SAM.015',
       'VRP.SAM.009', 'VRP.SAM.017', 'VRP.SAM.023', 'VRP.SAM.026',
       'VRP.SAM.004', 'VRP.SAM.014', 'VRP.SAM.003', 'VRP.SAM.018'],
      dtype=object)

In [8]:
import pandas as pd

def definir_pattern(row):
    dmc = str(row['DMC']).strip() if pd.notna(row['DMC']) else ''
    rap = str(row['RAP']).strip() if pd.notna(row['RAP']) else ''
    
    # 🔹 1ª prioridade: se tiver DMC
    if dmc == 'DMC.RCE.001':
        return 'VZ1.DMC.RCE.001'
    elif dmc == 'DMC.RCE.002':
        return 'VZ1.DMC.RCE.002'
    
    # 🔹 2ª prioridade: se DMC está vazio, olha o RAP
    if rap == 'RAP.RCE.001':
        return 'VZ1.AAT.RCE.010'
    elif rap == 'REL.RCE.001':
        return 'VZ1.AAT.RCE.030'
    elif rap in ['REQ.GAM.001/REQ.GAM.002', 'RAP.RF2.001']:
        return 'VZ1.EBO.GAM.001'
    elif rap == '' or rap.lower() == 'nan':
        return 'VZ1.AAT.RCE.010'
    
    # 🔹 Se não cair em nenhum caso, mantém o RAP original
    return rap

# Aplica no dataframe
Nos['Pattern'] = Nos.apply(definir_pattern, axis=1)

print(Nos[['DMC','RAP','Pattern']].head(20))


   DMC          RAP          Pattern
0       REL.RCE.001  VZ1.AAT.RCE.030
1       REL.RCE.001  VZ1.AAT.RCE.030
2       REL.RCE.001  VZ1.AAT.RCE.030
3       REL.RCE.001  VZ1.AAT.RCE.030
4       REL.RCE.001  VZ1.AAT.RCE.030
5       REL.RCE.001  VZ1.AAT.RCE.030
6       REL.RCE.001  VZ1.AAT.RCE.030
7       REL.RCE.001  VZ1.AAT.RCE.030
8       REL.RCE.001  VZ1.AAT.RCE.030
9       REL.RCE.001  VZ1.AAT.RCE.030
10      REL.RCE.001  VZ1.AAT.RCE.030
11      REL.RCE.001  VZ1.AAT.RCE.030
12      REL.RCE.001  VZ1.AAT.RCE.030
13      REL.RCE.001  VZ1.AAT.RCE.030
14      REL.RCE.001  VZ1.AAT.RCE.030
15      REL.RCE.001  VZ1.AAT.RCE.030
16      RAP.RCE.001  VZ1.AAT.RCE.010
17      RAP.RCE.001  VZ1.AAT.RCE.010
18      RAP.RCE.001  VZ1.AAT.RCE.010
19      RAP.RCE.001  VZ1.AAT.RCE.010


In [8]:
Nos.columns

Index(['Unnamed: 0', 'NODENUM', 'Consumo_Original', 'Consumo CANF', 'Consumo',
       'Join_Count_x', 'perda', 'perda_ajustada', 'Demanda', 'Join_Count_y',
       ...
       'created__4', 'last_edi_4', 'last_edi_5', 'TAG_VAZA_1', 'Zonapres_1',
       'ZonaMano_1', 'Gerencia_1', 'Validado_1', 'Longitude', 'Latitude'],
      dtype='object', length=108)

In [13]:
apoio['UDA'].unique()

array(['UDA.SAM.003', 'UDA.SAM.004', 'UDA.SAM.002', 'UDA.SAM.005',
       'UDA.SAM.001', ' ', 'UDA.TAG.003', 'UDA.ARQ.001'], dtype=object)

In [15]:
def definir_pattern_zona(row):
    zona = str(row['Zonapressa']).strip()
    uda = str(row['UDA']).strip()
    dmc = str(row['DMC']).strip()

    if zona == 'VRP.SAM.016':
        return 'VZ1.DMC.SAM.004'
    elif zona == 'VRP.SAM.014':
        return 'VZ1.DMC.SAM.003'
    elif dmc =='DMC.SAM.002':
        return 'VZ1.DMC.SAM.002'
    elif uda == 'UDA.SAM.002':
        return 'VZ1.AAT.SAM.050'
    elif uda == 'UDA.SAM.001':
        return 'VZ1.AAT.SAM.030'
    elif dmc == 'DMC.SAM.001':
        return 'VZ1.DMC.SAM.001'
    else:
        return 'VZ1.RAP.SAM.002'

# Nos['Pattern'] = Nos.apply(definir_pattern_zona, axis=1)
apoio['Pattern'] = apoio.apply(definir_pattern_zona, axis=1)

In [64]:
df_junctions['ID'] = df_junctions['ID'].astype(str)
Nos['NODENUM'] = Nos['NODENUM'].astype(str)


In [16]:
df_junctions

,ID,Elev,Demand,Pattern,Descricao
0,0,1055.21,0.0,VZ1.RAP.SAM.002,
1,1,1055.06,0.0,VZ1.DMC.SAM.002,
2,2,1156.50,0.0,VZ1.DMC.SAM.002,
3,3,1156.50,0.0,VZ1.RAP.SAM.002,
4,4,1156.50,0.0,VZ1.RAP.SAM.002,
...,...,...,...,...,...
21477,21496,1239.70,0.0,VZ1.AAT.SAM.050,
21478,21497,1239.70,0.0,VZ1.AAT.SAM.050,
21479,PM.VRP.SAM.017,1090.00,0.0,VZ1.RAP.SAM.002,
21480,488,1182.60,0.0,VZ1.RAP.SAM.002,Fechado pela ZP - cadastrar VMA


In [17]:
def definir_pattern_zona(row):
    dmc = str(row['DMC']).strip()
    uda = str(row['UDA']).strip()
    tag = str(row['TAG_VAZAO']).strip()
    
    # ==========================
    # PRIORIDADE 1: DMC
    # ==========================
    if dmc == 'DMC.GAM.003':
        return 'VZ1.DMC.GAM.003'
    elif dmc == 'DMC.GAM.004':
        return 'VZ1.DMC.GAM.004'
    elif dmc == 'DMC.GAM.005':
        return 'VZ1.DMC.GAM.005'
    
    # ==========================
    # PRIORIDADE 2: UDA
    # ==========================
    elif uda == 'UDA.GAM.002':
        return 'VZ1.AAT.GAM.070'
    elif uda == 'UDA.GAM.003':
        return 'VZ1.AAT.GAM.050'
    elif uda == 'UDA.GAM.004':
        return 'VZ1.AAT.GAM.090'
    elif uda == 'UDA.GAM.001':
        return 'VZ1.AAT.GAM.031'
    
    # ==========================
    # PRIORIDADE 3: TAG_VAZÃO
    # ==========================
    elif tag not in ['nan', 'None', '', 'NaN']:
        return tag
    
    # ==========================
    # DEFAULT
    # ==========================
    # else:
    #     return 'VZ1.AAT.GAM.031'


Nos['Pattern'] = Nos.apply(definir_pattern_zona, axis=1)

In [18]:
# Nos['Pattern'].value_counts()
apoio['Pattern'].value_counts()

Pattern
VZ1.RAP.SAM.002    9139
VZ1.AAT.SAM.050    5186
VZ1.AAT.SAM.030    3119
VZ1.DMC.SAM.004    1647
VZ1.DMC.SAM.002    1200
VZ1.DMC.SAM.001     947
VZ1.DMC.SAM.003     244
Name: count, dtype: int64

In [19]:
df_junctions = df_junctions.drop(columns=['Pattern'])

In [20]:
df_junctions

,ID,Elev,Demand,Descricao
0,0,1055.21,0.0,
1,1,1055.06,0.0,
2,2,1156.50,0.0,
3,3,1156.50,0.0,
4,4,1156.50,0.0,
...,...,...,...,...
21477,21496,1239.70,0.0,
21478,21497,1239.70,0.0,
21479,PM.VRP.SAM.017,1090.00,0.0,
21480,488,1182.60,0.0,Fechado pela ZP - cadastrar VMA


In [80]:
Nos.columns

Index(['Unnamed: 0', 'NODENUM', 'Consumo', 'Join_Count_x', 'perda', 'RAP_x',
       'perda_ajustada', 'Demanda_Final', 'Demanda', 'Join_Count_y',
       'TARGET_FID', 'ASSETGROUP', 'ASSETTYPE', 'tiposistem', 'designtype',
       'codunidade', 'lifecycles', 'material', 'diameter', 'Shape__Len',
       'rugosidade', 'posicionam', 'tipodesenh', 'contratoob', 'FROMDEVICE',
       'TODEVICETE', 'dataimplan', 'installdat', 'inserviced', 'retireddat',
       'notes', 'creator', 'creationda', 'updatedby', 'lastupdate', 'GLOBALID',
       'ORIG_FID', 'ORIG_SEQ', 'X_INI', 'Y_INI', 'X_FIM', 'Y_FIM', 'Sistema',
       'Localidade', 'RAP_y', 'UDA', 'DMC', 'BOOSTER', 'VRP', 'created_us',
       'created_da', 'last_edite', 'last_edi_1', 'TAG_VAZAO', 'Zonapressa',
       'ZonaManobr', 'GerenciaMa', 'Validado', 'COTA', 'POINT_X', 'POINT_Y',
       'POINT_Z', 'POINT_M', 'Longitude', 'Latitude', 'Pattern'],
      dtype='object')

In [69]:
df_junctions['ID'] = df_junctions['ID'].astype(str)
Nos['NODENUM'] = Nos['NODENUM'].astype(str)

df_junctions = df_junctions.merge(
    Nos[['NODENUM', 'Pattern']],
    how='left',
    left_on='ID',
    right_on='NODENUM'
)

In [21]:
df_junctions['ID'] = df_junctions['ID'].astype(str)
apoio['ID'] = apoio['ID'].astype(str)

df_junctions = df_junctions.merge(
    apoio[['ID', 'Pattern']],
    how='left',
    left_on='ID',
    right_on='ID'
)

In [ ]:
df_junctions = df_junctions.merge(
    Nos[['NODENUM', 'Pattern']],
    how='left',
    left_on='ID',
    right_on='NODENUM'
)

In [16]:
df_junctions = (
    df_junctions
    .drop(columns=["NODENUM", "Pattern"])   # remove as colunas antigas
    .rename(columns={"DMC_1": "Pattern"})   # renomeia DMC_1
)


In [22]:
df_junctions.columns

Index(['ID', 'Elev', 'Demand', 'Descricao', 'Pattern'], dtype='object')

In [24]:
# df_junctions['Pattern'].fillna()

In [25]:
df_junctions['Pattern'].unique()

array(['VZ1.DMC.SAM.002', 'VZ1.RAP.SAM.002', 'VZ1.DMC.SAM.001',
       'VZ1.AAT.SAM.050', 'VZ1.DMC.SAM.004', 'VZ1.AAT.SAM.030',
       'VZ1.DMC.SAM.003'], dtype=object)

In [71]:
df_junctions = df_junctions.drop(columns=["NODENUM"]) 

In [22]:
# dicionário de substituições
mapa_subs = {
    'DMC.SSB.011': 'VZ3.RAP.SSB.002',
    'DMC.SSB.012': 'VZ1.EBO.SSB.002',
    'DMC.SSB.013': ' ',
    'DMC.SSB.014': 'VZ1.EBO.SSB.005',
    'DMC.SSB.015': 'VZ3.RAP.SSB.002',
    None: ' '  # caso venha como None
}

# aplica substituições
df_junctions['Pattern'] = df_junctions['Pattern'].replace(mapa_subs)

# substitui NaN por espaço vazio
df_junctions['Pattern'] = df_junctions['Pattern'].fillna(' ')


In [72]:
df_junctions.columns

Index(['ID', 'Elev', 'Demand', 'Descricao', 'Pattern'], dtype='object')

In [73]:
df_junctions['Pattern'].unique()

array([nan, 'VZ1.DMC.SAM.002', 'VZ1.RAP.SAM.002', 'VZ1.DMC.SAM.001',
       'VZ1.AAT.SAM.050', 'VZ1.DMC.SAM.004', 'VZ1.DMC.SAM.003'],
      dtype=object)

In [74]:
df_junctions['Pattern'] = df_junctions['Pattern'].fillna('VZ1.RAP.SAM.002')
# df_junctions['Pattern'] = df_junctions['Pattern_y'].fillna('VZ1.AAT.GAM.031')

# df_junctions = df_junctions.drop(columns=['Pattern_x', 'Pattern_y'])

In [75]:
df_junctions.columns

Index(['ID', 'Elev', 'Demand', 'Descricao', 'Pattern'], dtype='object')

In [25]:
df_junctions = df_junctions.drop(columns=['NODENUM_x', 'NODENUM_y'])

In [76]:
df_junctions['Pattern'].unique()

array(['VZ1.RAP.SAM.002', 'VZ1.DMC.SAM.002', 'VZ1.DMC.SAM.001',
       'VZ1.AAT.SAM.050', 'VZ1.DMC.SAM.004', 'VZ1.DMC.SAM.003'],
      dtype=object)

In [27]:
df_junctions[df_junctions['Pattern'].isna()]


,ID,Elev,Demand,Descricao,Pattern


In [35]:
Nos['Pattern'].value_counts()

Pattern
VZ1.RAP.SAM.002    12601
VZ1.AAT.SAM.050     5186
VZ1.DMC.SAM.004     1674
VZ1.DMC.SAM.002     1201
VZ1.DMC.SAM.001      587
VZ1.DMC.SAM.003      249
Name: count, dtype: int64

In [10]:
vazio= Nos[Nos['Pattern']==' ']
vazio

,Unnamed: 0,NODENUM,Consumo,Join_Count_x,perda,perda_ajustada,Demanda,Join_Count_y,TARGET_FID,ASSETGROUP,...,created_da,last_edite,last_edi_1,TAG_VAZAO,Zonapressa,ZonaManobr,GerenciaMa,Shape_Leng,Shape_Area,Pattern


In [26]:
def update_junctions_inp(arquivo_inp, df_junctions):

    with open(arquivo_inp, 'r', encoding='utf-8') as f:
        linhas = f.readlines()

    nova_secao_junctions = []
    coletando = False

    for linha in linhas:
        linha_strip = linha.strip()

        if linha_strip.startswith("[JUNCTIONS]"):
            coletando = True
            nova_secao_junctions.append("[JUNCTIONS]\n")
            nova_secao_junctions.append(";ID               Elev            Demand            Pattern         ;Descricao\n")
            continue

        elif coletando and (linha_strip.startswith("[") and not linha_strip.startswith("[JUNCTIONS]")):
            coletando = False

            # 🔥 escreve com descrição
            for _, row in df_junctions.iterrows():

                descricao = (
                    f";{row['Descricao']}"
                    if 'Descricao' in df_junctions.columns and pd.notna(row['Descricao']) and row['Descricao'] != ""
                    else ""
                )

                nova_linha = (
                    f"{str(row['ID']): <18}"
                    f"{row['Elev']: <16.5f}"
                    f"{row['Demand']: <16.6f}"
                    f"{str(row['Pattern']): <14}"
                    f"{descricao}\n"
                )

                nova_secao_junctions.append(nova_linha)

        if not coletando:
            nova_secao_junctions.append(linha)

    with open(arquivo_inp, 'w', encoding='utf-8') as f:
        f.writelines(nova_secao_junctions)

In [27]:
arquivo_inp =  r"Epanet\Samambaia\SAM V6 - SUCESSSUL Padrao.inp"
update_junctions_inp(arquivo_inp, df_junctions=df_junctions)

In [91]:
arquivo_inp =  r"Epanet\Gama 2\GAM2 V11 copy.inp"
atualizar_junctions(arquivo_inp, df_junctions=df_junctions)

✅ Arquivo .inp atualizado com sucesso!


In [ ]:
update_junctions_inp(arquivo_inp, df_junctions=df_junctions)

In [19]:
lista_nos['NODENUM'] = lista_nos['NODENUM'].astype(str)
# df2['ID'] = df2['ID'].astype(int)


NameError: name 'lista_nos' is not defined

In [32]:
Nos['Pattern'].value_counts()

Pattern
VZ1.RAP.GAM.002    17787
VZ1.DMC.SAM.004     1674
VZ1.DMC.SAM.002     1201
VZ1.DMC.SAM.001      587
VZ1.DMC.SAM.003      249
Name: count, dtype: int64

In [43]:
lista_nos = Nos[['NODENUM','Pattern']]
lista_nos['NODENUM'] = lista_nos['NODENUM'].astype(str)

C:\Users\filipe_silva\AppData\Local\Temp\ipykernel_56244\964079145.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lista_nos['NODENUM'] = lista_nos['NODENUM'].astype(str)


In [44]:
df_merge = pd.merge(df_junctions, lista_nos,left_on='ID',right_on='NODENUM', how="left").copy()


In [45]:
df_merge["Demand"].fillna(0,inplace=True)

C:\Users\filipe_silva\AppData\Local\Temp\ipykernel_56244\771686823.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_merge["Demand"].fillna(0,inplace=True)


In [46]:
df_merge#.columns

,ID,Elev,Demand,Pattern_x,NODENUM,Pattern_y
0,0,1258.30,0.000000,;,0,VZ1.AAT.RCE.030
1,1,1258.83,0.001984,;,1,VZ1.AAT.RCE.030
2,2,1259.39,0.000000,;,2,VZ1.AAT.RCE.030
3,3,1260.00,0.000000,;,3,VZ1.AAT.RCE.030
4,4,1260.00,0.000000,;,4,VZ1.AAT.RCE.030
...,...,...,...,...,...,...
16054,7474,1260.00,0.000000,;,7474,VZ1.AAT.RCE.010
16055,7443,0.00,0.000000,;,7443,VZ1.AAT.RCE.030
16056,7476,0.00,0.000000,;,7476,VZ1.AAT.RCE.010
16057,7477,0.00,0.000000,;,7477,VZ1.AAT.RCE.010


In [47]:
# df_merge.drop(columns="Demand", inplace=True)
#df_merge.rename(columns={"Demanda":"Demand"}, inplace=True)

df_merge.drop(columns=["Pattern_x", "NODENUM"], inplace=True)
df_merge.rename(columns={"Pattern_y":"Pattern"}, inplace=True)


In [49]:
# Substituir os NaN por "VZ1.AAT.RCE.010"
df_merge['Pattern'] = df_merge['Pattern'].fillna('VZ1.AAT.RCE.010')

In [50]:
df_merge.isna().sum()

ID         0
Elev       0
Demand     0
Pattern    0
dtype: int64

In [51]:
df_merge['Pattern'].unique()

array(['VZ1.AAT.RCE.030', 'VZ1.AAT.RCE.010', 'VZ1.DMC.RCE.002',
       'VZ1.EBO.GAM.001', 'VZ1.DMC.RCE.001'], dtype=object)

In [ ]:
# df_merge.drop(columns="Pattern_x", inplace=True)
# df_merge.rename(columns={"Pattern_y":"Pattern"}, inplace=True)

In [52]:
df_merge["Demand"] = df_merge["Demand"].round(6)

In [38]:
df_junctions['Pattern'].value_counts()

Pattern
VZ1.AAT.RCE.010    7639
VZ1.AAT.RCE.030    5872
VZ1.EBO.GAM.001    2162
VZ1.DMC.RCE.003     352
                     12
Name: count, dtype: int64

In [34]:
df_merge=df_junctions

In [44]:
resultado_str = [str(x) for x in resultado]

df_merge.loc[df_merge["ID"].astype(str).isin(resultado_str), "Pattern"] = "PadraoVRP.RCE.002"



In [45]:
df_merge.loc[df_merge["Pattern"]==";", "Pattern"] = ''
df_merge

,ID,Elev,Demand,Pattern
0,0,1258.30,0.000000,VZ1.AAT.RCE.030
1,1,1258.83,0.001984,VZ1.AAT.RCE.030
2,2,1259.39,0.000000,VZ1.AAT.RCE.030
3,3,1260.00,0.000000,VZ1.AAT.RCE.030
4,4,1260.00,0.000000,VZ1.AAT.RCE.030
...,...,...,...,...
16032,2238,1214.50,0.000000,
16033,6261,1214.00,0.000000,
16034,6272,1213.00,0.000000,
16035,6273,1215.00,0.000000,


In [46]:
df_merge['Pattern'].value_counts()

Pattern
VZ1.AAT.RCE.010      6900
VZ1.AAT.RCE.030      5872
VZ1.EBO.GAM.001      2162
PadraoVRP.RCE.002     739
VZ1.DMC.RCE.003       352
                       12
Name: count, dtype: int64

In [47]:
df_merge.isna().value_counts()

ID     Elev   Demand  Pattern
False  False  False   False      16037
Name: count, dtype: int64

In [136]:
df_merge['Demand'] = df_merge['Demand'].fillna(0)

In [55]:
df_merge['Demand'].sum()

473.836327

In [56]:
# perda_ajustada['Demanda'].sum()

In [57]:
# perda_ajustada.to_excel('Tabelas para calibração\\Brazlândia\\chute Ana.xlsx')

In [58]:
# df_merge['Demand'].sum()

In [43]:
df_junctions[df_junctions['ID'].isin(lista_nos)]


,ID,Elev,Demand,Pattern
0,0,1258.30,0.000000,VZ1.SAT.RCE.031
1,1,1258.83,0.001984,VZ1.SAT.RCE.031
2,2,1259.39,0.000000,VZ1.SAT.RCE.031
3,3,1260.00,0.000000,VZ1.SAT.RCE.031
4,4,1260.00,0.000000,VZ1.SAT.RCE.031
...,...,...,...,...
16029,2235,1215.00,0.000000,VZ1.SAT.RCE.031
16030,2236,1216.50,0.000000,VZ1.SAT.RCE.031
16031,2237,1215.00,0.000000,VZ1.SAT.RCE.031
16032,2238,1214.50,0.000000,VZ1.SAT.RCE.031


In [26]:
df_junctions

,ID,Elev,Demand,Pattern
0,1,960.16,0.32,VZ3.RAP.SSB.002
1,2,906.35,0.09,DMC.SSB.009/10
2,3,904.41,0.12,DMC.SSB.009/10
3,4,923.23,0.00,VZ3.RAP.SSB.002
4,5,920.27,0.16,VZ3.RAP.SSB.002
...,...,...,...,...
7356,183,960.06,0.00,DMC.SSB.006/007
7357,184,965.00,0.00,DMC.SSB.006/007
7358,156,1140.00,0.00,DMC.SSB.009/10
7359,159,1140.00,0.00,DMC.SSB.009/10


In [27]:
df_junctions['Pattern'].unique()

array(['VZ3.RAP.SSB.002', 'DMC.SSB.009/10', 'DMC.SSB.003/4',
       'DMC.SSB.006/007', 'DMC.SSB.001', 'VZ1.EBO.SSB.002', 'DMC.SSB.005',
       'DMC.SSB.008', 'DMC.SSB.002', ' ', 'VZ1.EBO.SSB.005'], dtype=object)

In [47]:
Nos.columns

Index(['Unnamed: 0', 'NODENUM', 'Consumo', 'Join_Count_x', 'perda',
       'perda_ajustada', 'Demanda', 'Join_Count_y', 'TARGET_FID', 'ASSETGROUP',
       'ASSETTYPE', 'FROMDEVICE', 'TODEVICETE', 'GLOBALID', 'creationda',
       'creator', 'lastupdate', 'updatedby', 'installdat', 'notes', 'diameter',
       'lifecycles', 'inserviced', 'retireddat', 'material', 'designtype',
       'codunidade', 'posicionam', 'contratoob', 'tiposistem', 'tipodesenh',
       'rugosidade', 'dataimplan', 'Shape__Len', 'ORIG_FID', 'ORIG_SEQ',
       'X_inicial', 'Y_inicial', 'X_final', 'Y_final', 'POINT_X', 'POINT_Y',
       'POINT_Z', 'POINT_M', 'Cota', 'Sistema', 'Localidade', 'RAP', 'UDA',
       'DMC', 'BOOSTER', 'VRP', 'created_us', 'created_da', 'last_edite',
       'last_edi_1', 'TAG_VAZAO', 'Zonapressa', 'ZonaManobr', 'GerenciaMa',
       'Validado'],
      dtype='object')

In [49]:
# Garantir que as colunas estejam como string
Nos['NODENUM'] = Nos['NODENUM'].astype(str)
df_junctions['ID'] = df_junctions['ID'].astype(str)

# 1️⃣ Filtrar todos os nós com codunidade = 'SAT.RCE.013'
lista_nos = Nos.loc[Nos['codunidade'] == 'SAT.RCE.013', 'NODENUM'].tolist()

# 2️⃣ Atualizar o pattern no df_juctions para esses IDs
df_junctions.loc[df_junctions['ID'].isin(lista_nos), 'Pattern'] = 'VZ1.AAT.RCE.010'

# 3️⃣ Conferir o resultado
print(df_junctions.loc[df_junctions['ID'].isin(lista_nos), ['ID', 'Pattern']].head(10))
print(f"Total de linhas atualizadas: {df_junctions['Pattern'].eq('VZ1.AAT.RCE.010').sum()}")


      ID          Pattern
170  170  VZ1.AAT.RCE.010
171  171  VZ1.AAT.RCE.010
172  172  VZ1.AAT.RCE.010
173  173  VZ1.AAT.RCE.010
683  683  VZ1.AAT.RCE.010
684  684  VZ1.AAT.RCE.010
685  685  VZ1.AAT.RCE.010
686  686  VZ1.AAT.RCE.010
687  687  VZ1.AAT.RCE.010
688  688  VZ1.AAT.RCE.010
Total de linhas atualizadas: 7509


In [29]:
df_junctions

,ID,Elev,Demand,Pattern
0,1,960.16,0.32,VZ3.RAP.SSB.002
1,2,906.35,0.09,DMC.SSB.009/10
2,3,904.41,0.12,DMC.SSB.009/10
3,4,923.23,0.00,VZ3.RAP.SSB.002
4,5,920.27,0.16,VZ3.RAP.SSB.002
...,...,...,...,...
7356,183,960.06,0.00,DMC.SSB.006/007
7357,184,965.00,0.00,DMC.SSB.006/007
7358,156,1140.00,0.00,DMC.SSB.009/10
7359,159,1140.00,0.00,DMC.SSB.009/10


In [30]:
def encontrar_erros_utf8(caminho):
    with open(caminho, "rb") as f:  # lê em modo binário
        for i, linha in enumerate(f, start=1):
            try:
                linha.decode("utf-8")  # tenta decodificar
            except UnicodeDecodeError as e:
                print(f"❌ Erro na linha {i}: {e}")


In [32]:
encontrar_erros_utf8("Epanet\\São Sebastião\\Sao Sebastiao V10.inp")


In [33]:
arquivo_inp =  r"Epanet\\São Sebastião\\Sao Sebastiao V10.inp"
update_junctions_inp(arquivo_inp, df_junctions=df_junctions)

In [ ]:
# Usar map() para atualizar a coluna 'Demanda' com base no ID
df_junctions['Demand'] = df_junctions['ID'].map(lista_nos.set_index('FID')['Demand'])

# Exibir as primeiras linhas para conferência
print(df_junctions.head())


KeyError: 'Demand'

In [429]:
df_junctions.set_index("ID", inplace=True)  # Define "ID" como índice
df_junctions.update(perda_ajustada.set_index("ID"))  # Atualiza os valores
df_junctions.reset_index(inplace=True)  # Restaura "ID" como coluna normal

KeyError: "None of ['ID'] are in the columns"